# Workshop 2 — Exploring the Silver Layer with Pandas
**Team 4 · Tropical Cyclone Monitoring · Data Analysis Programming 2026-II**

This notebook reads the Parquet files produced by the `silver_processing` DAG in `datalake_silver/`
(`tropical_cyclones_YYYYMMDD_HHMMSS.parquet`) and shows their schema, lineage, content and quality flags.

**How to run** (no Python needed on the host — it runs inside the Airflow container):
```bash
docker compose exec airflow-scheduler jupyter nbconvert --to notebook --execute --inplace /opt/airflow/notebooks/02_silver_exploration.ipynb
```
It also runs locally with `pip install pandas pyarrow matplotlib jupyter` from the repository root.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
SILVER = ROOT / "datalake_silver"
files = sorted(SILVER.glob("tropical_cyclones_*.parquet"))
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
print(f"{len(files)} Silver files in {SILVER}")
pd.DataFrame([{"file": f.name, "size_KB": round(f.stat().st_size / 1024, 1),
               "rows": pq.ParquetFile(f).metadata.num_rows} for f in files])

## 1. Schema of a Silver file
The schema is enforced with an explicit pyarrow schema; categorical fields are dictionary-encoded and come back as `category` in Pandas.

In [ ]:
first = next(f for f in files if pq.ParquetFile(f).metadata.num_rows > 0)
print(first.name)
print(pq.read_schema(first).to_string(show_schema_metadata=False))

## 2. Lineage: which Bronze files produced each Silver file
Each file carries a `tcmon_lineage` metadata entry, and every row has `bronze_file` and `bronze_sha256` columns.

In [ ]:
lineage = [json.loads(pq.read_schema(f).metadata[b"tcmon_lineage"]) for f in files]
pd.DataFrame([{"silver_file": l["silver_file"], "bronze_files": ", ".join(l["bronze_files"]), "rows": l["rows"],
               "rejected": l["rejected"], **{f"dedup_{k}": v for k, v in l["dedup"].items()}} for l in lineage])

## 3. Read all Silver files with Pandas

In [ ]:
df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
print(df.shape)
print(df.dtypes.to_string())
df.head(10)

## 4. Content by source and storm

In [ ]:
(df.groupby(["source", "storm_id", "storm_name", "basin"], dropna=False, observed=True)
   .agg(rows=("event_id", "size"), first=("timestamp_utc", "min"), last=("timestamp_utc", "max"),
        max_wind_kt=("wind_kt", "max"), peak_wind_kt=("peak_wind_kt", "max"))
   .reset_index())

## 5. Missing values and quality flags

In [ ]:
nulls = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
display(nulls[nulls > 0].to_frame("null_%"))
display(df[["is_unnamed", "is_provisional", "is_spur", "motion_derived", "wind_iqr_outlier",
            "speed_zscore_outlier", "is_revision"]].sum().to_frame("rows_flagged"))
df["qc_flags"].replace("", "(none)").value_counts()

## 6. Tracks and intensity

In [ ]:
fixes = df[df.event_type == "tc_track_fix"].sort_values("timestamp_utc")
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for (sid, name), g in fixes.groupby(["storm_id", "storm_name"], dropna=False, observed=True):
    label = name if isinstance(name, str) else f"{sid} (unnamed)"
    ax[0].plot(g.longitude, g.latitude, marker=".", label=label)
    ax[1].plot(g.timestamp_utc, g.wind_kt, marker=".", label=label)
alerts = df[df.event_type == "tc_alert_episode"]
ax[0].scatter(alerts.longitude, alerts.latitude, marker="*", s=120, c="k", label="GDACS alert position")
ax[0].set(xlabel="Longitude (°)", ylabel="Latitude (°)", title="Track fixes (IBTrACS) and GDACS positions")
ax[1].set(xlabel="Time (UTC)", ylabel="Wind (kt)", title="Maximum sustained wind (1-min)")
ax[0].legend(fontsize=8); ax[1].tick_params(axis="x", rotation=30); plt.tight_layout(); plt.show()

## 7. Rejected records (quarantine)

In [ ]:
rej = sorted((SILVER / "_rejected").glob("*.json"))
print(f"{len(rej)} rejected-record files", [p.name for p in rej])